# Exact-Hessian GPU operator profile

Standalone Colab diagnostic for the production one-day collocation workload on `feature/issue-124/fuse-collocation-callbacks`.

It runs only three IPOPT iterations, captures a real warmed exact-Hessian callback and its IPOPT arguments, then replays that callback once under `torch.profiler`. Solver timing is intentionally not benchmarked here.

Select **Runtime → Change runtime type → GPU** before running all cells.

In [ ]:
# --- Install the profiled branch --------------------------------------------
import subprocess
import sys

import torch

REF = "feature/issue-124/fuse-collocation-callbacks"
REPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA Colab runtime is required.")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        f"git+{REPO_URL}@{REF}",
    ],
    check=True,
)

props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# --- Build the production profiling workload and capture a real Hessian call --
import datetime
import functools
import importlib.util
from pathlib import Path

import numpy as np
from dateutil import tz
import twin4build as tb
import twin4build.estimator._casadi_ipopt as ipopt
import twin4build.examples as examples_package
import twin4build.examples.utils as example_utils

# Load the same model callback used by full_workflow_example without running main().
example_path = Path(examples_package.__file__).parent / "full_workflow_example.py"
spec = importlib.util.spec_from_file_location("_hessian_profile_workflow", example_path)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)


def build_model():
    model = tb.Model(id="hessian_profile")
    model.load(
        semantic_model_filename=example_utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=workflow.fcn,
    )
    model.to("cuda", torch.float64)
    return model


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc = c["office_temperature_heating_controller"]
    cc = c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    c = model.components
    return [
        (c["office_valve_position_sensor"], 0.05 / 2),
        (c["office_temperature_sensor"], 0.1 / 2),
        (c["office_damper_position_sensor"], 0.05 / 2),
        (c["office_co2_sensor"], 30 / 2),
    ]


captured = {"calls": 0}
original_solve = ipopt.solve_ipopt_constrained


def capture_solve(
    x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jac_rows, jac_cols,
    options=None, *, hess_vals=None, hess_rows=None, hess_cols=None,
    early_stopping=None, print_level=0, quiet=True,
):
    if hess_vals is None:
        raise RuntimeError("The workload did not provide an exact Hessian callback.")

    @functools.wraps(hess_vals)
    def capture_hessian(*args):
        value = hess_vals(*args)
        captured["calls"] += 1
        if captured["calls"] == 2:
            captured["callback"] = hess_vals
            captured["args"] = tuple(
                np.array(arg, copy=True) if isinstance(arg, np.ndarray) else arg
                for arg in args
            )
        return value

    return original_solve(
        x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jac_rows, jac_cols,
        options=options,
        hess_vals=capture_hessian,
        hess_rows=hess_rows,
        hess_cols=hess_cols,
        early_stopping=early_stopping,
        print_level=print_level,
        quiet=quiet,
    )


ipopt.solve_ipopt_constrained = capture_solve
model = build_model()
estimator = tb.Estimator(tb.Simulator(model))
start = datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))
end = start + datetime.timedelta(hours=24)

result = estimator.estimate(
    parameters=build_parameters(model),
    measurements=build_measurements(model),
    start_time=[start],
    end_time=[end],
    step_size=1200,
    n_warmup=20,
    method=("casadi", "ipopt", "ad", "collocation"),
    options={
        "maxiter": 3,
        "exact_hessian": True,
        "early_stopping": False,
        "boundary_state_init": "rollout",
    },
)
ipopt.solve_ipopt_constrained = original_solve

if "callback" not in captured:
    raise RuntimeError(f"Expected at least two Hessian calls; observed {captured['calls']}.")
print(f"Captured Hessian call 2 of {captured['calls']} from a real IPOPT solve.")

In [ ]:
# --- Warm replay, measure, and profile exactly one Hessian call --------------
import statistics
import time
from IPython.display import FileLink, display

hessian = captured["callback"]
hessian_args = captured["args"]

# One out-of-band warmup ensures profiler startup is not confused with callback
# initialization. Then report ordinary synchronized wall time separately.
hessian(*hessian_args)
torch.cuda.synchronize()
wall_samples = []
for _ in range(3):
    torch.cuda.synchronize()
    started = time.perf_counter()
    hessian(*hessian_args)
    torch.cuda.synchronize()
    wall_samples.append(time.perf_counter() - started)

activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
torch.cuda.synchronize()
with torch.profiler.profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_flops=True,
) as prof:
    hessian(*hessian_args)
    torch.cuda.synchronize()

trace_path = "/content/twin4build_hessian_trace.json"
prof.export_chrome_trace(trace_path)

print(f"Unprofiled Hessian wall samples: {wall_samples}")
print(f"Median unprofiled Hessian wall time: {statistics.median(wall_samples):.6f} s")
print("\nTop operators by self CUDA time:\n")
print(
    prof.key_averages(group_by_input_shape=True).table(
        sort_by="self_cuda_time_total",
        row_limit=50,
    )
)
print("\nTop operators by self CPU time:\n")
print(
    prof.key_averages(group_by_input_shape=True).table(
        sort_by="self_cpu_time_total",
        row_limit=30,
    )
)
print("\nChrome trace (open with chrome://tracing or Perfetto):")
display(FileLink(trace_path))